# From Exposure to Action
### A Data-Driven Career Compass for the AI Era

**Will AI replace human jobs, and which occupations are most at risk of AI-driven displacement, transformation, or augmentation — and what should students and working professionals do about it?**

This notebook is the Phase 1 analysis deliverable. See `docs/From_Exposure_to_Action_Data_Dictionary.xlsx` for the full research framework (typology, hypotheses, target variables, guardrails) and `README.md` for the project overview.

**Status: data acquisition (AIOE, O*NET) and the AIOE+O*NET master join are implemented and runnable end to end. Remaining sources (ILO, OECD, BLS OEWS, GPTs are GPTs, Anthropic Economic Index, Statistics Canada) are still TODO — see `src/data_acquisition/README.md`. RQ sections below will fill in as each source is added.**

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 1. Data acquisition

Run once (or re-run to refresh) before this notebook can execute end to end:

```bash
python ../src/data_acquisition/fetch_aioe.py
python ../src/data_acquisition/fetch_onet.py
python ../src/data_acquisition/fetch_ilo_genai.py
python ../src/data_acquisition/fetch_bls_crosswalks.py
```

(Remaining fetchers -- OECD, BLS OEWS, GPTs are GPTs, Anthropic Economic Index, Statistics Canada -- are tracked in `src/data_acquisition/README.md`.)

## 2. Build the master occupation-level dataset

Joins AIOE + O*NET + ILO on occupation code via `src/crosswalks/` and `src/analysis/build_master_dataset.py`. Produces the Composite Exposure Score and a first-pass (provisional) Impact Pattern classification -- see that script's module docstring for the classification rule and its caveats.

In [ ]:
import sys
sys.path.insert(0, "../src")
from analysis.build_master_dataset import merge_all, save

# Run once (or re-run after adding a new source) to rebuild data/processed/master_occupations.csv.
# Requires fetch_aioe.py and fetch_onet.py to have been run first (see Section 1).
master = merge_all()
save(master)
master.head()

## 3. RQ1 — Consolidated risk ranking across indices

Top/bottom occupations by composite AI exposure. **Currently uses AIOE + ILO only** (the two indices joined so far in `build_master_dataset.py`) -- OECD and GPTs-are-GPTs are tracked as `[ ] later pass` in `src/data_acquisition/README.md` and will extend the correlation matrix once added. Tests H1 with the sources on hand; not the full picture yet.

In [ ]:
master = pd.read_csv("../data/processed/master_occupations.csv")

N = 15
CATEGORY_COLORS = {
    "Automation": "#2a78d6",       # blue
    "Transformation": "#eb6834",   # orange
    "Augmentation": "#1baf7a",     # aqua
}
MISSING_COLOR = "#9a9993"  # occupations with no impact_pattern yet

ranked = master.dropna(subset=["composite_exposure_score"]).sort_values(
    "composite_exposure_score", ascending=False
)
print(f"{len(ranked)} of {len(master)} occupations have a composite exposure score.")

display_cols = ["soc_code", "occupation_title", "composite_exposure_score",
                "aioe_score", "ilo_score", "impact_pattern"]
top_n = ranked.head(N)[display_cols]
bottom_n = ranked.tail(N)[display_cols]

print(f"\nTop {N} highest-exposure occupations:")
display(top_n)
print(f"\nBottom {N} lowest-exposure occupations:")
display(bottom_n)

In [ ]:
# Do the two available indices agree on which occupations are exposed?
# AIOE (Felten/Raj/Seamans, LLM-linguistic-task-based) and ILO (Gmyrek et al.,
# GPT-4o/Gemini task-rating-based) use different methodologies, so agreement here
# is informative even with just these two -- a low/insignificant rho would say the
# indices disagree more than expected, which matters as much as a high one.
paired = master.dropna(subset=["aioe_score", "ilo_score"])
rho, p_value = stats.spearmanr(paired["aioe_score"], paired["ilo_score"])
print(f"Spearman rho (AIOE vs ILO), n={len(paired)}: {rho:.3f} (p={p_value:.2e})")

print("\nimpact_pattern distribution (occupations with a value):")
print(master["impact_pattern"].value_counts())
print(f"({master['impact_pattern'].isna().sum()} occupations have no impact_pattern -- "
      "no ILO task-level match, see build_master_dataset.py)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_df = top_n.iloc[::-1]  # reverse so the highest-exposure occupation plots on top
colors = plot_df["impact_pattern"].map(CATEGORY_COLORS).fillna(MISSING_COLOR)
ax.barh(plot_df["occupation_title"], plot_df["composite_exposure_score"], color=colors)
ax.set_xlabel("Composite exposure score (z-scored mean of AIOE + ILO)")
ax.set_title(f"Top {N} occupations by AI exposure")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CATEGORY_COLORS.values()]
ax.legend(handles, CATEGORY_COLORS.keys(), title="Impact pattern", loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for pattern, color in CATEGORY_COLORS.items():
    subset = paired[paired["impact_pattern"] == pattern]
    ax.scatter(subset["aioe_score"], subset["ilo_score"], s=20, alpha=0.6,
               color=color, label=pattern)
no_pattern = paired[paired["impact_pattern"].isna()]
if len(no_pattern):
    ax.scatter(no_pattern["aioe_score"], no_pattern["ilo_score"], s=20, alpha=0.4,
               color=MISSING_COLOR, label="(no impact_pattern)")
ax.set_xlabel("AIOE score")
ax.set_ylabel("ILO GenAI exposure score")
ax.set_title(f"AIOE vs. ILO exposure (Spearman ρ={rho:.2f}, n={len(paired)})")
ax.legend(title="Impact pattern")
plt.tight_layout()
plt.show()

**Reading this section:** the ranking above is associative, not predictive -- a high composite score describes an occupation's task content as measured by two LLM-exposure methodologies today, not a forecast of job loss (see the project's guardrails in `README.md`). The Spearman correlation is the key sanity check: if AIOE and ILO disagreed strongly (rho near 0 or negative) it would say the two methodologies are capturing different things and the composite score should be read cautiously until OECD and GPTs-are-GPTs are added to triangulate further. Revisit both the ranking and this correlation once those sources join the master table.

**A caveat on ties in the ranking:** the ISCO-08 -> SOC 2018 crosswalk is many-to-many, so several distinct SOC titles here can trace back to the very same ISCO-08 unit group and inherit its exact `ilo_score`. When one of those tied occupations also has no AIOE coverage (`aioe_score` is NaN), its `composite_exposure_score` is driven entirely by that one shared ILO value -- so an exact tie in the top/bottom list reflects a shared ISCO parent, not two independent methodologies agreeing on that specific occupation. Worth noting explicitly in the write-up rather than presenting every entry as independently confirmed.

## 4. RQ2 -- Automation vs. Transformation vs. Augmentation

`impact_pattern` in the master table is now populated from ILO's task-level score variance (see `build_master_dataset.py`'s module docstring for the classification rule). This is a first-pass, provisional heuristic -- cross-check it here against the Anthropic Economic Index's own automation/augmentation usage split once that source is added (tests H2).

In [ ]:
# RQ2 cross-check: does the ILO-task-variance-based impact_pattern label
# (Automation/Transformation/Augmentation) actually line up with how people
# are using Claude on those occupations in the real world (Anthropic
# Economic Index's own automation/augmentation usage split)? This is a
# genuine cross-check of a provisional heuristic against an independent
# real-usage signal -- distinct from RQ5, which correlated the CONTINUOUS
# composite_exposure_score; this looks at the CATEGORICAL impact_pattern
# groups directly.

rq2 = master.dropna(subset=["impact_pattern"]).copy()

group_summary = rq2.groupby("impact_pattern").agg(
    n_occupations=("soc_code", "count"),
    n_with_anthropic_data=("anthropic_1p_automation_share", "count"),
    mean_automation_share=("anthropic_1p_automation_share", "mean"),
    mean_augmentation_share=("anthropic_1p_augmentation_share", "mean"),
).round(2)
group_summary = group_summary.reindex(["Automation", "Transformation", "Augmentation"])
print("=== Anthropic usage shares by impact_pattern group ===")
display(group_summary)


In [ ]:
order = ["Automation", "Transformation", "Augmentation"]
groups_automation = [rq2.loc[rq2["impact_pattern"] == p, "anthropic_1p_automation_share"].dropna() for p in order]
groups_augmentation = [rq2.loc[rq2["impact_pattern"] == p, "anthropic_1p_augmentation_share"].dropna() for p in order]

# Kruskal-Wallis (non-parametric, no normality assumption -- same reasoning
# as this project's use of Spearman correlation elsewhere) tests whether the
# three impact_pattern groups' usage-share distributions differ at all.
if all(len(g) >= 3 for g in groups_automation):
    h_auto, p_auto = stats.kruskal(*groups_automation)
    print(f"Kruskal-Wallis, anthropic_1p_automation_share across impact_pattern groups: H={h_auto:.2f}, p={p_auto:.3g}")
else:
    print("Not enough data in one or more groups for Kruskal-Wallis (automation_share).")

if all(len(g) >= 3 for g in groups_augmentation):
    h_aug, p_aug = stats.kruskal(*groups_augmentation)
    print(f"Kruskal-Wallis, anthropic_1p_augmentation_share across impact_pattern groups: H={h_aug:.2f}, p={p_aug:.3g}")
else:
    print("Not enough data in one or more groups for Kruskal-Wallis (augmentation_share).")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
rng = np.random.default_rng(0)

for ax, col, title in zip(
    axes,
    ["anthropic_1p_automation_share", "anthropic_1p_augmentation_share"],
    ["Automation-bucket share", "Augmentation-bucket share"],
):
    data_by_group = [rq2.loc[rq2["impact_pattern"] == p, col].dropna() for p in order]
    bp = ax.boxplot(data_by_group, tick_labels=order, patch_artist=True, showfliers=False, widths=0.5)
    for patch, pattern in zip(bp["boxes"], order):
        patch.set_facecolor(CATEGORY_COLORS[pattern])
        patch.set_alpha(0.7)
    for median_line in bp["medians"]:
        median_line.set_color("black")
    for i, (pattern, series) in enumerate(zip(order, data_by_group), start=1):
        jitter = rng.uniform(-0.15, 0.15, size=len(series))
        ax.scatter(
            np.full(len(series), i) + jitter, series, color=CATEGORY_COLORS[pattern],
            alpha=0.3, s=15, edgecolor="none",
        )
    ax.set_title(title)
    ax.set_xlabel("impact_pattern")

axes[0].set_ylabel("Anthropic 1P API usage share (%)")
fig.suptitle("Does impact_pattern predict actual Claude usage pattern? (RQ2)")
plt.tight_layout()
plt.savefig("../outputs/figures/rq2_impact_pattern_vs_anthropic_usage.png", dpi=150)
plt.show()


**Reading this section:** `impact_pattern` is a provisional heuristic derived from ILO's task-level score variance (see `build_master_dataset.py`'s docstring) -- it was never fit or validated against real usage data. This cross-check asks a direct question: do occupations labeled "Automation" actually show a higher real-world automation-bucket usage share than occupations labeled "Augmentation" or "Transformation"? The Kruskal-Wallis test (non-parametric, no normality assumption) checks whether the three groups' usage-share distributions differ at all; the boxplots show the shape of that difference directly.

If the groups don't separate cleanly, that is itself a genuine finding about this heuristic's current limits, not a bug -- worth stating plainly in the write-up rather than smoothed over, and consistent with RQ5's finding that the continuous `composite_exposure_score` also didn't correlate with automation-style usage in the naively-expected direction.


## 5. RQ3 — Task-level vulnerability

Which tasks within an occupation carry the highest ILO exposure scores.

In [ ]:
# RQ3: which TASKS within an occupation carry the highest ILO exposure
# scores? Uses load_ilo_task_detail() -- genuine task-level granularity
# (score_2025 varies per task), NOT the occupation-level aggregate
# (mean_score_2025) used elsewhere in this notebook. Standalone /
# illustrative, same pattern as the Canada appendix -- not joined into
# the master table, since task-level rows don't fit a one-row-per-
# occupation table.

from analysis.build_master_dataset import load_ilo_task_detail

task_detail = load_ilo_task_detail()
occ_scores = task_detail.drop_duplicates("isco_code")[
    ["isco_code", "occupation_title", "occupation_mean_score_2025"]
]
top_occupations = occ_scores.sort_values("occupation_mean_score_2025", ascending=False).head(6)
print("=== Top occupations by ILO mean exposure score (2025) -- task breakdown below ===")
display(top_occupations)


In [ ]:
n_panels = len(top_occupations)
ncols = 3
nrows = -(-n_panels // ncols)  # ceil
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5 * nrows))
axes = np.atleast_1d(axes).flatten()

for ax, (_, row) in zip(axes, top_occupations.iterrows()):
    occ_tasks = task_detail[task_detail["isco_code"] == row["isco_code"]].sort_values(
        "task_score_2025", ascending=True
    ).tail(8)
    labels = [d[:45] + ("\u2026" if len(d) > 45 else "") for d in occ_tasks["task_description"]]
    ax.barh(labels, occ_tasks["task_score_2025"], color="#2a78d6", alpha=0.85)
    ax.set_title(f"{row['occupation_title']} (ISCO {row['isco_code']})", fontsize=10)
    ax.set_xlabel("ILO task exposure score (2025)")
    ax.tick_params(axis="y", labelsize=8)

for ax in axes[n_panels:]:
    ax.axis("off")

fig.suptitle("RQ3 \u2014 Highest-exposure tasks within the most-exposed occupations", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/figures/rq3_task_level_vulnerability.png", dpi=150)
plt.show()


**Reading this section:** these are the 6 occupations with the highest *occupation-level* ILO score, each broken down into its own *task-level* scores (top 8 tasks shown, longest bar = most exposed task within that occupation). Within any one occupation, tasks typically span a real range -- that spread is exactly what feeds this project's `impact_pattern` classification (high task-to-task variance → "Transformation"; low variance + high mean → "Automation"; low variance + low/mid mean → "Augmentation").

Task descriptions are truncated for display; the full text is available from `load_ilo_task_detail()`'s `task_description` column if a specific task needs to be quoted in the write-up. These scores are ILO's own 2025 GPT-4o/Gemini-updated estimates -- treat individual task rankings as illustrative of *within-occupation* spread, not as precise, individually-validated measurements.


## 6. RQ4 — Protective skill profile

Compare O*NET skill/ability distributions between high- and low-exposure occupation groups (tests H4).

In [ ]:
# RQ4: which O*NET skills/abilities are more (or less) common in
# high-exposure vs. low-exposure occupations -- a "protective skill
# profile" a person could read as: which skills correlate with
# occupations that are currently LESS exposed on the theoretical indices?
# Splits on composite_exposure_score TERTILES (top third = high, bottom
# third = low) rather than impact_pattern, since impact_pattern is
# categorical/provisional and exposure here needs a continuous ranking to
# split cleanly into comparison groups.

skill_cols = [c for c in master.columns if c.startswith("skill__")]

rq4 = master.dropna(subset=["composite_exposure_score"]).copy()
low_cut = rq4["composite_exposure_score"].quantile(1 / 3)
high_cut = rq4["composite_exposure_score"].quantile(2 / 3)
rq4["exposure_group"] = np.select(
    [rq4["composite_exposure_score"] <= low_cut, rq4["composite_exposure_score"] >= high_cut],
    ["Low exposure", "High exposure"],
    default="Mid (excluded)",
)
rq4_ends = rq4[rq4["exposure_group"] != "Mid (excluded)"]
print(
    f"{len(rq4_ends)} occupations in the low/high exposure comparison "
    f"({(rq4_ends['exposure_group'] == 'Low exposure').sum()} low, "
    f"{(rq4_ends['exposure_group'] == 'High exposure').sum()} high)"
)

# Mann-Whitney U per skill (non-parametric two-group comparison, same
# reasoning as Kruskal-Wallis/Spearman used elsewhere in this notebook),
# plus the mean difference so the table is readable without staring at
# only p-values.
rows = []
for col in skill_cols:
    low_vals = rq4_ends.loc[rq4_ends["exposure_group"] == "Low exposure", col].dropna()
    high_vals = rq4_ends.loc[rq4_ends["exposure_group"] == "High exposure", col].dropna()
    if len(low_vals) < 3 or len(high_vals) < 3:
        continue
    u, p = stats.mannwhitneyu(high_vals, low_vals, alternative="two-sided")
    rows.append({
        "skill": col.replace("skill__", "").replace("_", " "),
        "mean_low_exposure": round(low_vals.mean(), 2),
        "mean_high_exposure": round(high_vals.mean(), 2),
        "diff_high_minus_low": round(high_vals.mean() - low_vals.mean(), 2),
        "mannwhitney_p": p,
    })
skill_diff = pd.DataFrame(rows).sort_values("diff_high_minus_low")
print("\n=== Skill importance: high-exposure minus low-exposure occupations ===")
display(skill_diff)


In [ ]:
fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(skill_diff))))
colors = ["#2a78d6" if v < 0 else "#eb6834" for v in skill_diff["diff_high_minus_low"]]
ax.barh(skill_diff["skill"], skill_diff["diff_high_minus_low"], color=colors, alpha=0.85)
ax.axvline(0, color="#3d3d3a", linewidth=1)
ax.set_xlabel("Mean importance, high-exposure minus low-exposure occupations")
ax.set_title("Which O*NET skills separate high- vs. low-exposure occupations? (RQ4)")
plt.tight_layout()
plt.savefig("../outputs/figures/rq4_skill_profile_diff.png", dpi=150)
plt.show()


In [ ]:
# Most "protective" skills: largest NEGATIVE diff = most associated with
# LOW-exposure occupations (i.e. more important where AI exposure is lower).
most_protective = skill_diff.sort_values("diff_high_minus_low").head(5)
most_exposed_associated = skill_diff.sort_values("diff_high_minus_low", ascending=False).head(5)
print("=== Skills most associated with LOW-exposure occupations (candidate 'protective' skills) ===")
display(most_protective)
print("\n=== Skills most associated with HIGH-exposure occupations ===")
display(most_exposed_associated)


**Reading this section:** this compares O*NET skill-importance ratings between occupations in the top third and bottom third of `composite_exposure_score` -- it does not claim a skill *causes* lower exposure, only that it's more common in occupations that currently score lower on the theoretical indices. A skill showing up here as "protective" is a candidate worth investigating for the professional/reskilling layer later, not a guarantee.

The Mann-Whitney p-values flag which differences are unlikely to be noise given this dataset's size; with ~35 O*NET skill/ability elements tested at once, treat any single borderline p-value cautiously (multiple-comparisons risk) and look for a *pattern* across related skills rather than one skill in isolation.


## 7. RQ5 — Theoretical exposure vs. realized labor-market impact

Correlate composite exposure score against BLS OEWS multi-year employment/wage growth, and against Anthropic Economic Index actual-usage share (tests H3). Associative only — not a forecast.

In [ ]:
# RQ5: does the THEORETICAL composite exposure score line up with what's
# already visible in the real labor market? Two independent real-world
# signals: (1) BLS OEWS employment growth 2023->2025 (does employment
# already show contraction in high-exposure occupations?), and (2) the
# Anthropic Economic Index's actual usage share (are people already using
# Claude to automate/augment the same occupations the theoretical indices
# flag as exposed?). Associative only, per this project's guardrails -- not
# a forecast, and BLS OEWS growth compresses many macro factors (not just
# AI) into one number over a short 2-year window.

rq5 = master.dropna(subset=["composite_exposure_score"]).copy()


def spearman_report(df, x_col, y_col, label):
    sub = df.dropna(subset=[x_col, y_col])
    if len(sub) < 3:
        print(f"{label}: not enough overlapping data (n={len(sub)})")
        return None
    rho, p = stats.spearmanr(sub[x_col], sub[y_col])
    print(f"{label}: rho={rho:.3f}, p={p:.3g}, n={len(sub)}")
    return rho, p, len(sub)


print("=== RQ5 correlations: composite_exposure_score vs. real-world signals ===\n")
growth_result = spearman_report(
    rq5, "composite_exposure_score", "employment_growth_rate",
    "composite_exposure_score vs. employment_growth_rate (BLS OEWS 2023->2025)",
)
auto_result = spearman_report(
    rq5, "composite_exposure_score", "anthropic_1p_automation_share",
    "composite_exposure_score vs. anthropic_1p_automation_share",
)
aug_result = spearman_report(
    rq5, "composite_exposure_score", "anthropic_1p_augmentation_share",
    "composite_exposure_score vs. anthropic_1p_augmentation_share",
)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for pattern, color in CATEGORY_COLORS.items():
    subset = rq5[(rq5["impact_pattern"] == pattern) & rq5["employment_growth_rate"].notna()]
    ax.scatter(
        subset["composite_exposure_score"], subset["employment_growth_rate"],
        label=pattern, color=color, alpha=0.7, edgecolor="white", linewidth=0.4, s=40,
    )
missing = rq5[rq5["impact_pattern"].isna() & rq5["employment_growth_rate"].notna()]
if len(missing):
    ax.scatter(
        missing["composite_exposure_score"], missing["employment_growth_rate"],
        label="impact_pattern missing", color=MISSING_COLOR, alpha=0.5,
        edgecolor="white", linewidth=0.4, s=40,
    )
ax.axhline(0, color="#9a9993", linewidth=1, linestyle="--")
ax.set_xlabel("Composite exposure score (standardized)")
ax.set_ylabel("Employment growth rate, 2023\u21922025 (annualized)")
ax.set_title("Theoretical exposure vs. realized employment growth (RQ5)")
ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig("../outputs/figures/rq5_exposure_vs_employment_growth.png", dpi=150)
plt.show()


In [ ]:
high_exposure_threshold = rq5["composite_exposure_score"].quantile(2 / 3)
high_exp = rq5[rq5["composite_exposure_score"] >= high_exposure_threshold].dropna(
    subset=["employment_growth_rate"]
)

declining = high_exp[high_exp["employment_growth_rate"] < 0].sort_values("employment_growth_rate")
growing = high_exp[high_exp["employment_growth_rate"] >= 0].sort_values(
    "employment_growth_rate", ascending=False
)

print(f"Of {len(high_exp)} high-exposure occupations with employment data:")
print(f"  {len(declining)} already show DECLINING employment (2023-2025) -- exposure + realized contraction")
print(f"  {len(growing)} still show GROWING employment -- exposure not yet visible, or buffered by augmentation\n")

cols = ["soc_code", "occupation_title", "composite_exposure_score", "impact_pattern", "employment_growth_rate"]
print("=== High exposure + declining employment (top 10) ===")
display(declining[cols].head(10))

print("\n=== High exposure + still growing (top 10) ===")
display(growing[cols].head(10))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for pattern, color in CATEGORY_COLORS.items():
    subset = rq5[(rq5["impact_pattern"] == pattern) & rq5["anthropic_1p_automation_share"].notna()]
    ax.scatter(
        subset["composite_exposure_score"], subset["anthropic_1p_automation_share"],
        label=pattern, color=color, alpha=0.7, edgecolor="white", linewidth=0.4, s=40,
    )
missing = rq5[rq5["impact_pattern"].isna() & rq5["anthropic_1p_automation_share"].notna()]
if len(missing):
    ax.scatter(
        missing["composite_exposure_score"], missing["anthropic_1p_automation_share"],
        label="impact_pattern missing", color=MISSING_COLOR, alpha=0.5,
        edgecolor="white", linewidth=0.4, s=40,
    )
ax.set_xlabel("Composite exposure score (standardized)")
ax.set_ylabel("Anthropic 1P API automation-bucket share (%)")
ax.set_title("Theoretical exposure vs. actual Claude usage pattern (RQ5)")
ax.legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig("../outputs/figures/rq5_exposure_vs_anthropic_usage.png", dpi=150)
plt.show()


**Reading this section:** two independent real-world signals, tested against the same theoretical `composite_exposure_score`.

**Employment growth (BLS OEWS, 2023→2025):** essentially uncorrelated with `composite_exposure_score` (rho=0.025, p=0.47, n=817). Among the 269 high-exposure occupations with employment data, the split is almost exactly even -- 134 already declining vs. 135 still growing. This is consistent with the caveat above: two years compresses far more than AI adoption into one number (post-pandemic sector recovery, interest-rate-driven hiring/layoffs, demographic shifts, offshoring), and labor markets typically take years to fully reflect a technology shock. A near-zero correlation here does not invalidate the exposure framework; it means the framework is measuring *potential*, not yet-realized displacement -- which is the whole point of framing this as "theoretical exposure vs. realized impact" rather than "predicted vs. actual."

**Anthropic Economic Index usage shares:** here the real result is more interesting, and worth stating plainly rather than left generic. `composite_exposure_score` correlates *negatively* with `anthropic_1p_automation_share` (rho=−0.230, p=1.25e-08, n=598) and *positively*, by the same magnitude, with `anthropic_1p_augmentation_share` (rho=+0.230, same p, same n -- expected, since the two shares are near-complementary). In plain terms: occupations that score HIGHER on the theoretical exposure indices are, in real Claude usage today, more associated with augmentation-style interactions and less with automation-style ones -- the opposite of what a naive "high exposure score = high automation" reading would predict.

A few honest, non-exclusive readings, none confirmed by this data alone: real-world AI use may currently lag the full automation potential the theoretical indices capture; complex, higher-exposure tasks may in practice still benefit more from human-AI collaboration than fully automatable ones do; or there may be a selection effect in which occupations' workers use Claude directly and how. Whichever explanation holds, this is a genuinely useful finding for this project's decision-support framing -- it's a real-world data point *against* the "high theoretical exposure means imminent full automation" narrative, not for it.

Both comparisons are associative, not causal or predictive -- consistent with this project's decision-support framing (evidence → risks → opportunities), never "AI will replace you."


## 8. Student layer — education path vs. exposure

Join O*NET Job Zones (education/experience/training required) to the composite exposure score; aggregate by education level.

## 9. Professional layer — reskilling paths

Occupation-to-occupation skill-vector similarity to surface, for a given high-exposure occupation, the nearest lower-exposure alternatives and the skill gap to close.

## 10. Appendix — Canada in context

One comparison chart: Statistics Canada's own NOC-based AI exposure estimates against the equivalent global/US-ranked occupations. Not a parallel pipeline (see README guardrails).

## 11. Summary & what this means for you

Decision-support framing only — evidence, risks, opportunities, skills, and alternatives; never a single prescriptive recommendation. Sets up the Phase 2 web app's dashboard fields (see the Data Dictionary's "Phase 2 Tool Design" tab).